# CoRe gripper tool centres

![Every channel pair probing both CoRe gripper tools](img/core_gripper_tool_centers.png)

Each neighbouring pair probes both CoRe gripper tools from inside: three walls per tool, a circle
through them, its centre against the model. Two channels on the same tool should agree.

## Setup

Every channel needs a conductive tip.

In [ ]:
import asyncio

from pylabrobot.hamilton.star import STAR
from pylabrobot.resources import hamilton_96_tiprack_50uL, hamilton_tip_carrier_L5

star = STAR()
await star.setup()

tip_carrier = hamilton_tip_carrier_L5(name="tip_carrier")
tip_carrier[0] = hamilton_96_tiprack_50uL(name="tip_rack_50ul")
star.deck.assign_child_resource(tip_carrier, track=32)

## Tool tops

The stop discs of channels 0 and 1 find both tops at once, before any tip goes on.

In [ ]:
back_tool = star.get_resource("STAR_deck_core_grippers_back")
front_tool = star.get_resource("STAR_deck_core_grippers_front")
back_top = back_tool.get_location_wrt(other=star.deck, x="c", y="c", z="t")
front_top = front_tool.get_location_wrt(other=star.deck, x="c", y="c", z="t")

# channel 0 over the back tool's rim, channel 1 over the front tool's
await star.pipettes.move_to_safe_z()
await star.pipettes.move_to_xy_positions(
    x=back_top.x + 2.5,
    ys={0: back_top.y, 1: front_top.y},
    make_space=True,
)


async def probe_top(channel, top):
    return await star.pipettes.probe_z_using_clld(
        channel=channel,
        search_start_position=top.z + 6.0,
        search_end_position=top.z - 1.0,
        allow_without_tip=True,
        post_detection_distance=4.0,
    )


back_z, front_z = await asyncio.gather(
    probe_top(channel=0, top=back_top),
    probe_top(channel=1, top=front_top),
)
await star.pipettes.move_to_safe_z()

if back_z is None or front_z is None:
    raise RuntimeError("a tool top was not found")
modelled_z_top = round((back_z + front_z) / 2, 2)

## Every pair

`front_channel - 1` probes the back tool, `front_channel` the front one.

In [ ]:
tip_rack = star.get_resource("tip_rack_50ul")
all_channels = list(range(star.pipettes.num_channels))
await star.pipettes.pick_up_tips(tip_spots=tip_rack.column(0), use_channels=all_channels)

results = {}
for front_channel in all_channels[1:]:
    centers = await star.diagnostics.probe_core_gripper_tool_centers(
        front_channel=front_channel,
        modelled_z_top=modelled_z_top,
    )
    results[front_channel] = centers

await star.pipettes.return_tips()

## Result

Each channel's offset from the modelled centre, in mm:

In [ ]:
for front_channel, centers in results.items():
    back_channel = front_channel - 1
    for channel, tool, model in (
        (back_channel, "back", back_top),
        (front_channel, "front", front_top),
    ):
        dx = centers[channel].x - model.x
        dy = centers[channel].y - model.y
        print(f"channel {channel}, {tool} tool: dX {dx:+.2f}, dY {dy:+.2f}")

A channel that is off shows on both tools; a tool that is off shows for every channel. A tip's
runout adds to its channel's offset: take a fresh tip per pair to average it out. Align an off
channel by hand with [the adjustment procedure](../hardware/adjusting-robot.md).